## Q1. Bagging vs Boosting for Credit Default Prediction

### Introduction

In this question we compare two ensemble approaches — a Random Forest (bagging) and XGBoost (gradient boosting) — to predict whether a bank customer will experience a credit default. The dataset contains 6,000 customer records with 13 financial and demographic features. Both models are tuned via 5-fold cross-validation using log-loss as the optimisation criterion, then compared on accuracy, ROC-AUC, and their ability to generalise across repeated held-out splits.

In [ ]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, recall_score
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (needed for 3D projection)
import statsmodels.api as sm

# Path setup
credit_path = Path("./credit_default.csv")

# Dataframe setup
credit_df = pd.read_csv(credit_path)
X = credit_df.drop(columns=['default'])
y = credit_df['default']

print(f"Dataset: {credit_df.shape[0]} observations, {credit_df.shape[1] - 1} features")
print(f"Class balance -- Default: {y.mean():.2%} | No Default: {1 - y.mean():.2%}")
print(credit_df.head(3))

### Hyperparameter Tuning

Both models were evaluated using GridSearchCV with a 5-fold cross-validation scheme. Log-loss was used as the refitting criterion for both, consistent with its role as the splitting criterion in the underlying decision trees.

For the Random Forest, the grid explored 144 combinations across n_estimators (50, 200, 500, 1000), max_depth (2, 3, 5, or unconstrained), max_features (3, 5, or 9 — roughly 23%, 38%, and 69% of the 13 features), and min_samples_leaf (1, 5, 10).

For XGBoost, 128 combinations were tested over n_estimators (50, 200, 500, 1000), max_depth (2, 3, 4, 10), learning_rate (0.01, 0.03, 0.05, 0.1), and subsample (0.7 or 1.0).

Accuracy measures the fraction of correctly classified customers. In a credit-default context it can be misleading when classes are imbalanced, since a model can score high accuracy by mostly predicting the majority class. ROC-AUC summarises the model's ability to rank customers by default risk across all classification thresholds — a value closer to 1 indicates stronger discriminative ability regardless of class distribution.

#### Random Forest Findings

- Accuracy and AUC show diminishing returns as n_estimators increases: the biggest gains come from going from 50 to 200 trees, with improvements becoming negligible beyond 500. This makes sense — more trees reduce variance but can't fix bias.
- Deeper trees (max_depth of None or 5) tend to achieve slightly higher CV accuracy and AUC than shallow ones (max_depth = 2), as they can capture more complex decision boundaries.
- However, unconstrained depth produces noticeably higher train accuracy than CV accuracy, indicating overfitting — the forest is memorising training patterns that don't generalise. Shallower trees show a smaller train-CV gap and better regularisation, at some cost to bias.
- Increasing min_samples_leaf acts as an additional regulariser: larger values (e.g. 10) reduce overfitting but also reduce accuracy slightly, while min_samples_leaf = 1 allows the most flexible trees with the largest train-CV gap.
- The best RF model by log-loss and by accuracy/AUC likely share similar hyperparameters (deep trees, large ensemble, small min_samples_leaf), since log-loss is highly correlated with both metrics for well-calibrated classifiers.

#### XGBoost Findings

- The interaction between learning_rate and n_estimators is critical. A small learning rate (0.01) needs many more trees to converge; with only 50 trees the model underfits. A larger rate (0.1) converges faster but can overshoot with 1000 trees. The best configurations tend to pair a moderate learning rate with a larger ensemble.
- XGBoost shows a more pronounced train-CV accuracy gap than the Random Forest at large max_depth. This reflects its sequential boosting nature: each new tree aggressively corrects predecessor errors, making it more prone to overfitting on training data.
- Subsample values below 1.0 partially mitigate overfitting through stochastic sampling.

In [ ]:
# ============================================================
# 1.1: Hyperparameter Tuning -- Random Forest (5-Fold CV)
# ============================================================

rf_param_grid = {
    'n_estimators':     [50, 200, 500, 1000],
    'max_depth':        [2, 3, 5, None],         # None = grow until leaves are pure
    'max_features':     [3, 5, 9],               # ~sqrt(13), ~40%, ~70% of 13 features
    'min_samples_leaf': [1, 5, 10]
}

rf_base = RandomForestClassifier(criterion='log_loss', random_state=42, n_jobs=-1)

rf_grid = GridSearchCV(
    rf_base, rf_param_grid,
    cv=5,
    scoring=['neg_log_loss', 'accuracy', 'roc_auc'],
    refit='neg_log_loss',
    return_train_score=True,
    n_jobs=-1, verbose=1
)
rf_grid.fit(X, y)

rf_results  = pd.DataFrame(rf_grid.cv_results_)
best_rf_idx = rf_grid.best_index_

print(f"\n{'='*60}")
print("RANDOM FOREST -- Best Parameters (log-loss criterion):")
for k, v in rf_grid.best_params_.items():
    print(f"   {k}: {v}")
print(f"   CV Log-Loss: {-rf_grid.best_score_:.4f}")
print(f"   CV Accuracy: {rf_results.loc[best_rf_idx, 'mean_test_accuracy']:.4f}")
print(f"   CV AUC:      {rf_results.loc[best_rf_idx, 'mean_test_roc_auc']:.4f}")

# Top 5 by Accuracy and by AUC
_rf_cols_raw  = ['param_n_estimators', 'param_max_depth', 'param_max_features',
                 'param_min_samples_leaf', 'mean_test_accuracy', 'mean_test_roc_auc',
                 'mean_test_neg_log_loss']
_rf_cols_nice = ['n_est', 'max_depth', 'max_feat', 'min_leaf', 'CV Acc', 'CV AUC', 'CV NegLL']

rf_top_acc = rf_results.nlargest(5, 'mean_test_accuracy')[_rf_cols_raw].reset_index(drop=True)
rf_top_acc.columns = _rf_cols_nice
rf_top_auc = rf_results.nlargest(5, 'mean_test_roc_auc')[_rf_cols_raw].reset_index(drop=True)
rf_top_auc.columns = _rf_cols_nice

print(f"\nTop 5 RF by Accuracy:\n{rf_top_acc.round(4).to_string(index=False)}")
print(f"\nTop 5 RF by AUC:\n{rf_top_auc.round(4).to_string(index=False)}")

In [ ]:
# ============================================================
# 1.2: Hyperparameter Tuning -- XGBoost (5-Fold CV)
# ============================================================

xgb_param_grid = {
    'n_estimators':  [50, 200, 500, 1000],
    'max_depth':     [2, 3, 4, 10],       # 10 approximates None (XGBoost requires positive int)
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample':     [0.7, 1.0]
}

xgb_base = XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1, verbosity=0)

xgb_grid = GridSearchCV(
    xgb_base, xgb_param_grid,
    cv=5,
    scoring=['neg_log_loss', 'accuracy', 'roc_auc'],
    refit='neg_log_loss',
    return_train_score=True,
    n_jobs=-1, verbose=1
)
xgb_grid.fit(X, y)

xgb_results  = pd.DataFrame(xgb_grid.cv_results_)
best_xgb_idx = xgb_grid.best_index_

print(f"\n{'='*60}")
print("XGBOOST -- Best Parameters (log-loss criterion):")
for k, v in xgb_grid.best_params_.items():
    print(f"   {k}: {v}")
print(f"   CV Log-Loss: {-xgb_grid.best_score_:.4f}")
print(f"   CV Accuracy: {xgb_results.loc[best_xgb_idx, 'mean_test_accuracy']:.4f}")
print(f"   CV AUC:      {xgb_results.loc[best_xgb_idx, 'mean_test_roc_auc']:.4f}")

_xgb_cols_raw  = ['param_n_estimators', 'param_max_depth', 'param_learning_rate',
                  'param_subsample', 'mean_test_accuracy', 'mean_test_roc_auc',
                  'mean_test_neg_log_loss']
_xgb_cols_nice = ['n_est', 'max_depth', 'lr', 'subsample', 'CV Acc', 'CV AUC', 'CV NegLL']

xgb_top_acc = xgb_results.nlargest(5, 'mean_test_accuracy')[_xgb_cols_raw].reset_index(drop=True)
xgb_top_acc.columns = _xgb_cols_nice
xgb_top_auc = xgb_results.nlargest(5, 'mean_test_roc_auc')[_xgb_cols_raw].reset_index(drop=True)
xgb_top_auc.columns = _xgb_cols_nice

print(f"\nTop 5 XGB by Accuracy:\n{xgb_top_acc.round(4).to_string(index=False)}")
print(f"\nTop 5 XGB by AUC:\n{xgb_top_auc.round(4).to_string(index=False)}")

In [ ]:
# ============================================================
# 1.3: RF Analysis -- Accuracy & AUC vs n_estimators by max_depth
# ============================================================

# Convert None to string so groupby/labels work cleanly
rf_plot = rf_results.copy()
rf_plot['param_max_depth'] = rf_plot['param_max_depth'].astype(str)

rf_by_depth = rf_plot.groupby(
    ['param_max_depth', 'param_n_estimators'], dropna=False
).agg(
    cv_acc   =('mean_test_accuracy',  'mean'),
    cv_auc   =('mean_test_roc_auc',   'mean'),
    train_acc=('mean_train_accuracy', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Random Forest: CV Accuracy & AUC vs n_estimators\n(averaged over max_features, min_samples_leaf)', fontsize=12)

for depth, grp in rf_by_depth.groupby('param_max_depth'):
    grp = grp.sort_values('param_n_estimators')
    axes[0].plot(grp['param_n_estimators'], grp['cv_acc'], marker='o', label=f'max_depth={depth}')
    axes[1].plot(grp['param_n_estimators'], grp['cv_auc'], marker='o', label=f'max_depth={depth}')

for ax, ylabel, title in zip(
    axes,
    ['CV Accuracy', 'CV ROC-AUC'],
    ['Accuracy vs n_estimators', 'AUC vs n_estimators']
):
    ax.set_xlabel('n_estimators')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# RF: effect of min_samples_leaf (averaged over all other params)
rf_by_leaf = rf_plot.groupby('param_min_samples_leaf').agg(
    cv_acc   =('mean_test_accuracy',  'mean'),
    cv_auc   =('mean_test_roc_auc',   'mean'),
    train_acc=('mean_train_accuracy', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rf_by_leaf['param_min_samples_leaf'], rf_by_leaf['cv_acc'],    marker='o', label='CV Accuracy')
ax.plot(rf_by_leaf['param_min_samples_leaf'], rf_by_leaf['cv_auc'],    marker='s', label='CV AUC')
ax.plot(rf_by_leaf['param_min_samples_leaf'], rf_by_leaf['train_acc'], marker='^', linestyle='--', label='Train Accuracy')
ax.set_xlabel('min_samples_leaf')
ax.set_ylabel('Score')
ax.set_title('RF: Accuracy, AUC and Train Score vs min_samples_leaf')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 1.4: XGBoost Analysis -- Accuracy & AUC vs n_estimators by learning_rate
# ============================================================

xgb_by_lr = xgb_results.groupby(
    ['param_learning_rate', 'param_n_estimators']
).agg(
    cv_acc   =('mean_test_accuracy',  'mean'),
    cv_auc   =('mean_test_roc_auc',   'mean'),
    train_acc=('mean_train_accuracy', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('XGBoost: CV Accuracy & AUC vs n_estimators\n(averaged over max_depth, subsample)', fontsize=12)

for lr, grp in xgb_by_lr.groupby('param_learning_rate'):
    grp = grp.sort_values('param_n_estimators')
    axes[0].plot(grp['param_n_estimators'], grp['cv_acc'], marker='o', label=f'lr={lr}')
    axes[1].plot(grp['param_n_estimators'], grp['cv_auc'], marker='o', label=f'lr={lr}')

for ax, ylabel, title in zip(
    axes,
    ['CV Accuracy', 'CV ROC-AUC'],
    ['Accuracy vs n_estimators', 'AUC vs n_estimators']
):
    ax.set_xlabel('n_estimators')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 1.5: Overfitting Check -- Train vs CV Accuracy by max_depth
# ============================================================

rf_overfit = rf_plot.groupby('param_max_depth', dropna=False).agg(
    train_acc=('mean_train_accuracy', 'mean'),
    cv_acc   =('mean_test_accuracy',  'mean'),
).reset_index()
rf_overfit['gap'] = rf_overfit['train_acc'] - rf_overfit['cv_acc']

xgb_overfit = xgb_results.groupby('param_max_depth').agg(
    train_acc=('mean_train_accuracy', 'mean'),
    cv_acc   =('mean_test_accuracy',  'mean'),
).reset_index()
xgb_overfit['gap'] = xgb_overfit['train_acc'] - xgb_overfit['cv_acc']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Overfitting Check: Train vs CV Accuracy by max_depth', fontsize=13)

for ax, df, title in zip(axes, [rf_overfit, xgb_overfit], ['Random Forest', 'XGBoost']):
    x = np.arange(len(df))
    ax.bar(x - 0.2, df['train_acc'], 0.4, label='Train Accuracy', color='steelblue')
    ax.bar(x + 0.2, df['cv_acc'],    0.4, label='CV Accuracy',    color='salmon')
    ax.set_xticks(x)
    ax.set_xticklabels(df['param_max_depth'].astype(str))
    ax.set_xlabel('max_depth')
    ax.set_ylabel('Accuracy')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0.5, 1.05)

plt.tight_layout()
plt.show()

print("RF -- Train vs CV Accuracy gap by max_depth:")
print(rf_overfit[['param_max_depth', 'train_acc', 'cv_acc', 'gap']].round(4).to_string(index=False))
print("\nXGB -- Train vs CV Accuracy gap by max_depth:")
print(xgb_overfit[['param_max_depth', 'train_acc', 'cv_acc', 'gap']].round(4).to_string(index=False))

### Model Comparison

#### Best Model Selection

In general, the best model by log-loss and by accuracy may differ slightly — log-loss rewards well-calibrated probability estimates, while accuracy is threshold-dependent (defaulting to a 0.5 cutoff). Models with large, well-tuned ensembles tend to rank highly on all three metrics simultaneously.

#### Feature Importance

Both models expose a feature_importances_ attribute — for the Random Forest this is the mean decrease in impurity averaged across all trees, and for XGBoost it is the gain-based importance (average improvement in loss per split).

In a credit-risk context, we'd expect variables like past_default, num_late_payments_12m, debt_to_income, and credit_utilization to be strong predictors since they directly reflect financial stress. Whether the two models agree on the top predictors reflects signal consistency — high agreement suggests robust predictors, while disagreement may come from different importance definitions or correlated features where each model picks a different proxy.

If importance is highly concentrated on a few features, the remaining predictors add little value and the problem is essentially low-dimensional. A spread-out distribution suggests multiple complementary signals.

In [ ]:
# ============================================================
# 1.6: Model Comparison -- Top 5 Feature Importances
# ============================================================

feature_names = X.columns.tolist()

rf_imp  = pd.Series(rf_grid.best_estimator_.feature_importances_,  index=feature_names).sort_values(ascending=False)
xgb_imp = pd.Series(xgb_grid.best_estimator_.feature_importances_, index=feature_names).sort_values(ascending=False)

print(f"\nTop 5 Features -- Random Forest (best by log-loss):\n{rf_imp.head(5).round(4).to_string()}")
print(f"\nTop 5 Features -- XGBoost (best by log-loss):\n{xgb_imp.head(5).round(4).to_string()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Top 5 Feature Importances: RF vs XGBoost (best model by log-loss)', fontsize=13)

sns.barplot(x=rf_imp.head(5).values,  y=rf_imp.head(5).index,  ax=axes[0], palette='Blues_r')
axes[0].set_title('Random Forest')
axes[0].set_xlabel('Importance')

sns.barplot(x=xgb_imp.head(5).values, y=xgb_imp.head(5).index, ax=axes[1], palette='Oranges_r')
axes[1].set_title('XGBoost')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

# Full ranking side-by-side
rank_df = pd.DataFrame({
    'RF Importance':  rf_imp.round(4),
    'RF Rank':        range(1, len(feature_names) + 1),
    'XGB Importance': xgb_imp.reindex(rf_imp.index).round(4),
    'XGB Rank':       xgb_imp.rank(ascending=False).astype(int).reindex(rf_imp.index)
})
print(f"\nFull Feature Ranking Comparison:\n{rank_df.to_string()}")

### Final Evaluation

The best RF and XGB models (selected by log-loss) were evaluated over 30 repeated 80/20 train/test splits with different random seeds, giving a distribution of Accuracy and Recall that removes dependence on any single partition.

In a credit default setting, Recall (sensitivity) deserves higher priority than overall Accuracy. A false negative — predicting no default when the customer actually defaults — carries a much greater financial cost for the bank (unrecovered loans) than a false positive (declining a creditworthy customer). A model with higher recall catches more true defaulters even at the expense of some false alarms.

In [ ]:
# ============================================================
# 1.7: Final Evaluation -- 30 Repeated Train/Test Splits (80/20)
# ============================================================

best_rf_params  = dict(rf_grid.best_params_)
best_xgb_params = dict(xgb_grid.best_params_)

rf_accs, rf_recs, xgb_accs, xgb_recs = [], [], [], []

for seed in range(30):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

    rf_run = RandomForestClassifier(**best_rf_params, criterion='log_loss', random_state=42, n_jobs=-1)
    rf_run.fit(X_train, y_train)
    y_rf = rf_run.predict(X_test)
    rf_accs.append(accuracy_score(y_test, y_rf))
    rf_recs.append(recall_score(y_test, y_rf))

    xgb_run = XGBClassifier(**best_xgb_params, eval_metric='logloss', random_state=42, n_jobs=-1, verbosity=0)
    xgb_run.fit(X_train, y_train)
    y_xgb = xgb_run.predict(X_test)
    xgb_accs.append(accuracy_score(y_test, y_xgb))
    xgb_recs.append(recall_score(y_test, y_xgb))

print(f"\n{'='*60}")
print("FINAL EVALUATION -- 30 Repeated Train/Test Splits (80/20):")
print(f"   RF  Accuracy: {np.mean(rf_accs):.4f}  ±  {np.std(rf_accs):.4f}")
print(f"   XGB Accuracy: {np.mean(xgb_accs):.4f}  ±  {np.std(xgb_accs):.4f}")
print(f"   RF  Recall:   {np.mean(rf_recs):.4f}  ±  {np.std(rf_recs):.4f}")
print(f"   XGB Recall:   {np.mean(xgb_recs):.4f}  ±  {np.std(xgb_recs):.4f}")

models    = ['Random Forest', 'XGBoost']
acc_means = [np.mean(rf_accs),  np.mean(xgb_accs)]
acc_stds  = [np.std(rf_accs),   np.std(xgb_accs)]
rec_means = [np.mean(rf_recs),  np.mean(xgb_recs)]
rec_stds  = [np.std(rf_recs),   np.std(xgb_recs)]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
fig.suptitle('Final Evaluation: RF vs XGBoost over 30 Train/Test Splits', fontsize=14)

axes[0].bar(models, acc_means, yerr=acc_stds, capsize=10,
            color=['steelblue', 'darkorange'], alpha=0.85, width=0.4)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy (mean ± std)')
axes[0].set_ylim(max(0, min(acc_means) - 0.05), min(1, max(acc_means) + 0.05))
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(models, rec_means, yerr=rec_stds, capsize=10,
            color=['steelblue', 'darkorange'], alpha=0.85, width=0.4)
axes[1].set_ylabel('Recall')
axes[1].set_title('Recall (mean ± std)')
axes[1].set_ylim(max(0, min(rec_means) - 0.1), min(1, max(rec_means) + 0.1))
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Q3. PCA in Practice

### Introduction

In this question we explore the structure of a dataset recording occupational information for 1,200 workers using Principal Component Analysis (PCA). The dataset contains nine continuous predictors covering physical work demands (standing, lifting, manual intensity, repetitive motion), sedentary activities (seated hours, computer use), and socio-economic characteristics (age, income, meeting hours), alongside a binary target chronic_pain. The PCA section focuses exclusively on the explanatory variables to uncover the main axes of variation before incorporating the target in the logistic regression that follows.

---

### Data Inspection

The dataset has 1,200 observations and 9 features. The summary statistics show substantial scale heterogeneity across variables. Some notable differences:

- Age has a mean of about 40.5 and a standard deviation of ~10.6
- Annual income (GBP) has a mean around 28,953 and a std of ~7,998 — several orders of magnitude larger than the hourly work variables
- The physical work variables (standing, lifting, manual intensity, repetitive motion) have means roughly between 1.7 and 5.2, with stds between 1.3 and 2.4
- Meetings hours per week has a higher std (~4.2) compared to the other time-based measures
- Chronic pain affects about 27.9% of workers (roughly 1 in 4)

The correlation matrix highlights two broad clusters, which are confirmed by the PCA loadings:

- Physical/manual cluster: standing hours, lifting hours, manual intensity score, and repetitive motion score are all positively correlated — workers who stand more also tend to lift more and report higher manual intensity
- Desk-work cluster: seated hours, computer hours, and meetings hours are positively associated; these are largely orthogonal to the physical cluster, since active and sedentary work don't really coexist in a given working day

Age is relatively independent of both clusters, while income shows a mild association with the desk-work group (higher-income workers tend to be office-based).

In [ ]:
# ============================================================
# 3.1: Data Inspection
# ============================================================

workers_path = Path("./workers.csv")
workers_df   = pd.read_csv(workers_path)

feat_cols = [
    'age', 'annual_income_gbp', 'standing_hours_per_day', 'lifting_hours_per_day',
    'manual_intensity_score', 'repetitive_motion_score',
    'seated_hours_per_day', 'computer_hours_per_day', 'meetings_hours_per_week'
]
X_w = workers_df[feat_cols]
y_w = workers_df['chronic_pain']

print(f"Workers dataset: {workers_df.shape[0]} observations, {len(feat_cols)} features")
print(f"Chronic pain prevalence: {y_w.mean():.2%}\n")
print("Summary statistics:")
print(X_w.describe().round(2).to_string())

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(X_w.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix -- Workers Dataset', fontsize=13)
plt.tight_layout()
plt.show()

### PCA: Preprocessing and Justification

Before running PCA, the features were standardised (zero mean, unit variance) using StandardScaler. Two reasons justify this:

1. Scale differences: annual_income_gbp has a standard deviation several orders of magnitude larger than the hourly variables. Without standardisation, PCA would be dominated by income simply because of its larger numeric range, not because it's genuinely more variable in a meaningful sense.
2. PCA maximises variance: since PCA finds the directions of maximum variance, variables on larger scales would artificially dominate the first principal component. Standardisation puts all variables on equal footing so that PCA reflects the underlying covariance structure, not measurement units.

Mean-centering is performed automatically by StandardScaler and is also a formal requirement for PCA (the covariance matrix is defined around the mean).

In [ ]:
# ============================================================
# 3.2: PCA -- Standardise and Fit
# ============================================================

# Standardisation is required: annual_income_gbp (~20 000) dwarfs hours-per-day (~1–8);
# PCA maximises variance, so without scaling it would be dominated by income alone.
scaler  = StandardScaler()
X_w_std = scaler.fit_transform(X_w)

pca    = PCA()
X_pca  = pca.fit_transform(X_w_std)

eigvals   = pca.explained_variance_
var_ratio = pca.explained_variance_ratio_
cum_var   = np.cumsum(var_ratio)
n_feat    = len(feat_cols)
pc_labels = [f'PC{i+1}' for i in range(n_feat)]

print(f"\n{'='*60}")
print("PCA -- Variance Explained per Component:")
for i, (ev, vr, cv) in enumerate(zip(eigvals, var_ratio, cum_var)):
    print(f"  PC{i+1}: eigenvalue={ev:.3f}  |  var={vr:.2%}  |  cumulative={cv:.2%}")

### Principal Components: Eigenvalues and Loadings

#### Naming the Principal Components

Looking at the correlation loadings (the correlation between each standardised variable and each PC):

- PC1 — "Physical Labour Intensity": dominated by manual_intensity_score, lifting_hours_per_day, standing_hours_per_day, and repetitive_motion_score, all loading above 0.62. Desk-work variables have near-zero loadings here. PC1 essentially distinguishes workers in physically demanding jobs from everyone else.

- PC2 — "Office / Desk Work": driven by computer_hours_per_day (~0.70), meetings_hours_per_week (~0.62), annual_income_gbp (~0.60), and seated_hours_per_day (~0.58). PC2 captures the intensity of desk-based, collaborative, higher-paid work. It's orthogonal to PC1, which makes sense — physical and office work are largely independent occupational dimensions.

- PC3 — "Seniority / Age": almost entirely explained by age (loading ~0.92), with a secondary contribution from income (~0.31). This component captures career stage and earnings, largely independent of job type.

The correlation circle plots confirm these groupings visually — arrows for the four physical variables cluster together in the PC1 direction, desk-work arrows cluster in the PC2 direction, and age points strongly along PC3. Variables pointing at roughly right angles are uncorrelated with respect to those two PCs.

In [ ]:
# ============================================================
# 3.3: Factor Loadings Plots (Correlation Circles)
# ============================================================

# Correlation loadings: cor(x_i, PC_j) = component[j,i] * sqrt(eigenvalue[j])
# These lie within the unit circle for standardised inputs.
comp_loadings = pca.components_[:3]                         # shape (3, n_features)
corr_loadings = (comp_loadings.T * np.sqrt(eigvals[:3]))    # shape (n_features, 3)
loadings_df   = pd.DataFrame(corr_loadings, index=feat_cols, columns=['PC1', 'PC2', 'PC3'])

def _plot_loading_circle(pc_x, pc_y, ax):
    idx_x = int(pc_x[2:]) - 1
    idx_y = int(pc_y[2:]) - 1
    for feat in feat_cols:
        lx = loadings_df.loc[feat, pc_x]
        ly = loadings_df.loc[feat, pc_y]
        ax.annotate('', xy=(lx, ly), xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.5))
        ax.text(lx * 1.12, ly * 1.12, feat, fontsize=7.5, ha='center', va='center')
    ax.axhline(0, color='grey', lw=0.5, ls='--')
    ax.axvline(0, color='grey', lw=0.5, ls='--')
    circle = plt.Circle((0, 0), 1, fill=False, color='lightgrey', ls='--')
    ax.add_patch(circle)
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_xlabel(f'{pc_x}  ({var_ratio[idx_x]:.1%} var)', fontsize=10)
    ax.set_ylabel(f'{pc_y}  ({var_ratio[idx_y]:.1%} var)', fontsize=10)
    ax.set_title(f'Loadings: {pc_x} vs {pc_y}')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('PCA: Factor Loadings (Correlation Circles)', fontsize=13)
_plot_loading_circle('PC1', 'PC2', axes[0])
_plot_loading_circle('PC1', 'PC3', axes[1])
_plot_loading_circle('PC2', 'PC3', axes[2])
plt.tight_layout()
plt.show()

### 2D and 3D Projections

The 2D scatter plot of PC1 vs PC2 maps workers into a "job-type space" — workers in the upper-right are physically active and office-intensive, while those in the lower-left have low demands on both dimensions. The two chronic pain groups overlap substantially in this 2D projection, suggesting that job type alone (without age) doesn't cleanly separate pain cases from non-cases. That said, there may be some concentration of pain cases at the extremes of PC1 (heavy manual labour) or PC2 (intensive desk work), since both are associated with musculoskeletal stress through different mechanisms.

Adding PC3 in the 3D projection brings in the age/seniority axis. Older workers may show a slight tendency toward chronic pain at any level of PC1 or PC2, providing some additional separation that the first two PCs couldn't capture.

In [ ]:
# ============================================================
# 3.4: 2D and 3D Projections (coloured by chronic_pain)
# ============================================================

_pain_map = {0: ('steelblue', 'No Chronic Pain'), 1: ('tomato', 'Chronic Pain')}

fig, ax = plt.subplots(figsize=(8, 6))
for label, (color, name) in _pain_map.items():
    mask = y_w == label
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, alpha=0.5, s=20, label=name)
ax.set_xlabel(f'PC1  ({var_ratio[0]:.1%} var)')
ax.set_ylabel(f'PC2  ({var_ratio[1]:.1%} var)')
ax.set_title('PCA: 2D Projection  (PC1 vs PC2)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig = plt.figure(figsize=(10, 7))
ax3 = fig.add_subplot(111, projection='3d')
for label, (color, name) in _pain_map.items():
    mask = y_w == label
    ax3.scatter(X_pca[mask, 0], X_pca[mask, 1], X_pca[mask, 2],
                c=color, alpha=0.5, s=20, label=name)
ax3.set_xlabel(f'PC1  ({var_ratio[0]:.1%})')
ax3.set_ylabel(f'PC2  ({var_ratio[1]:.1%})')
ax3.set_zlabel(f'PC3  ({var_ratio[2]:.1%})')
ax3.set_title('PCA: 3D Projection  (PC1, PC2, PC3)')
ax3.legend()
plt.tight_layout()
plt.show()

### Dimensionality Reduction Criteria

Three standard criteria were applied:

1. Kaiser criterion (eigenvalue > 1): retain only components whose eigenvalue exceeds 1, meaning they explain more variance than a single standardised variable would on its own.
2. Scree plot (elbow method): plot eigenvalues in decreasing order and look for an "elbow" — the point where successive eigenvalues drop steeply before levelling off. Components before the elbow are retained.
3. Cumulative variance threshold: retain the minimum number of components needed to explain at least 80% (or 90%) of total variance.

The actual variance breakdown is roughly: PC1 explains 21.1%, PC2 adds another 17.8% (cumulative 38.9%), PC3 adds 11.6% (cumulative 50.4%), and the remaining components each contribute between 7% and 10%. It takes 7 components to cross the 80% threshold and 8 to reach 90%.

The three criteria give divergent recommendations:

- Kaiser criterion: retain 3 components (PC1 = 1.90, PC2 = 1.60, PC3 = 1.04, all above 1; PC4 = 0.86 falls below)
- Scree plot: the eigenvalues drop from 1.90 to 1.60 to 1.04 to 0.86 with no sharp elbow — the spectrum is relatively flat. The clearest change of slope is at PC4, which also points to retaining 3 components.
- 80% variance threshold: requires 7 components; 8 for 90%

The flat eigenvalue spectrum suggests variance is spread fairly evenly across dimensions without a dominant low-dimensional structure. This makes dimensionality reduction less effective here than in datasets with a few strong latent factors.

We retain 3 principal components, balancing interpretability and Kaiser/scree agreement, while accepting that they capture only ~50% of total variance. PC1 (physical labour), PC2 (desk work), and PC3 (age/seniority) each correspond to a clear occupational dimension, which justifies their use in the subsequent logistic regression. Retaining 7 components for the 80% threshold would largely eliminate any dimensionality reduction benefit and produce components that are difficult to interpret.

In [ ]:
# ============================================================
# 3.5: Dimensionality Reduction Criteria
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PCA: Dimensionality Reduction Criteria', fontsize=13)

# Scree plot
axes[0].plot(range(1, n_feat + 1), eigvals, marker='o', color='steelblue', lw=2)
axes[0].axhline(y=1, color='tomato', ls='--', lw=1.5, label='Kaiser criterion (λ = 1)')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Eigenvalue')
axes[0].set_title('Scree Plot')
axes[0].set_xticks(range(1, n_feat + 1))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative variance
axes[1].plot(range(1, n_feat + 1), cum_var * 100, marker='s', color='darkorange', lw=2)
axes[1].axhline(y=80, color='green',  ls='--', lw=1.5, label='80% threshold')
axes[1].axhline(y=90, color='tomato', ls='--', lw=1.5, label='90% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance Explained (%)')
axes[1].set_title('Cumulative Variance Explained')
axes[1].set_xticks(range(1, n_feat + 1))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

kaiser_n = int(np.sum(eigvals > 1))
pct80_n  = int(np.argmax(cum_var >= 0.80)) + 1
pct90_n  = int(np.argmax(cum_var >= 0.90)) + 1

print(f"\nKaiser criterion (λ > 1):  retain {kaiser_n} component(s)")
print(f"80% variance threshold:    retain {pct80_n} component(s)")
print(f"90% variance threshold:    retain {pct90_n} component(s)")

---

## Q3. Logistic Regression

### Data Inspection: chronic_pain and its Correlations

The target chronic_pain is binary, with a prevalence of 27.9% (335 out of 1,200 workers). The class imbalance is moderate and doesn't require resampling for logistic regression.

Correlation with original features (sorted by absolute value):

- lifting_hours_per_day is the most strongly correlated (~0.216)
- repetitive_motion_score follows (~0.185)
- standing_hours_per_day comes next (~0.167)
- manual_intensity_score (~0.154) and age (~0.093) are in the middle
- annual_income_gbp and computer_hours_per_day are the weakest correlates

Physical-work variables dominate, which is consistent with the PCA findings. Income and computer hours show the weakest linear relationship with chronic pain.

Correlation with the retained PCs:

- PC1 (Physical Labour) is by far the most predictive (~0.273)
- PC2 (Desk Work) and PC3 (Age/Seniority) show weaker but non-trivial associations (~0.089 and ~0.070 respectively)

PC1 being the strongest correlate is consistent with physical variables being the strongest individual predictors — the PCA essentially bundled them together into a single dimension.

In [ ]:
# ============================================================
# 3.6: Data Inspection -- chronic_pain correlations
# ============================================================

pain_corr_orig = X_w.corrwith(y_w).sort_values(key=abs, ascending=False)
print(f"\n{'='*60}")
print("Correlation of original features with chronic_pain:")
print(pain_corr_orig.round(3).to_string())

pc_named = pd.DataFrame(X_pca[:, :3], columns=['PC1', 'PC2', 'PC3'])
pain_corr_pca = pc_named.corrwith(y_w)
print(f"\nCorrelation of retained PCs with chronic_pain:")
print(pain_corr_pca.round(3).to_string())

### Logistic Regression with Original Variables

A full model was first fitted using all 9 features. Three predictors were not statistically significant at the 5% level — annual_income_gbp (p = 0.240), manual_intensity_score (p = 0.127), and computer_hours_per_day (p = 0.321). These are dropped, yielding a selected model with 6 predictors.

The selected model results (all 6 predictors significant at the 5% level):

- age: OR = 1.022 [1.009, 1.034], p = 0.001
- standing_hours_per_day: OR = 1.112 [1.031, 1.199], p = 0.006
- lifting_hours_per_day: OR = 1.320 [1.188, 1.468], p < 0.001
- repetitive_motion_score: OR = 1.143 [1.071, 1.220], p < 0.001
- seated_hours_per_day: OR = 1.089 [1.016, 1.166], p = 0.016
- meetings_hours_per_week: OR = 1.045 [1.012, 1.079], p = 0.006

McFadden Pseudo-R² = 0.079, AIC = 1322.70, 7 parameters (6 predictors + intercept).

Lifting_hours_per_day is the strongest predictor (OR = 1.32) — each additional hour of daily lifting is associated with a 32% increase in the odds of chronic pain, holding other variables constant. Repetitive_motion_score (OR = 1.14) and standing_hours_per_day (OR = 1.11) follow, all reflecting the well-established link between manual labour and musculoskeletal disorders. Seated_hours_per_day (OR = 1.09) and meetings_hours_per_week (OR = 1.04) are significant but with smaller effects.

Some variables are difficult to interpret jointly. Lifting hours, standing hours, manual intensity, and repetitive motion are all moderately correlated — they measure different facets of physical workload. In the full model, manual_intensity_score loses significance (p = 0.127) despite a decent univariate correlation with chronic pain (0.15), which is a classic sign of multicollinearity: once lifting and repetitive motion are in the model, manual intensity adds little new information since it shares most of its explanatory variance with them. Similarly, computer hours becomes redundant once seated hours and meetings hours are controlled for. Coefficients for correlated predictors can shift and become harder to interpret because their effects can't be cleanly separated.

In [ ]:
# ============================================================
# 3.7: Logistic Regression -- Original Variables
# ============================================================

# Full model (all 9 features)
X_orig_full = sm.add_constant(X_w)
logit_full   = sm.Logit(y_w, X_orig_full).fit(disp=0)
print(f"\n{'='*60}")
print("LOGISTIC REGRESSION -- Full model (9 original features):")
print(logit_full.summary2())

# Select significant predictors (p < 0.05, excluding intercept)
sig_feats = [v for v in logit_full.pvalues.index if v != 'const' and logit_full.pvalues[v] < 0.05]
print(f"\nSignificant predictors (p < 0.05): {sig_feats}")

# Refit with significant predictors only
X_orig_sel = sm.add_constant(X_w[sig_feats])
logit_sel   = sm.Logit(y_w, X_orig_sel).fit(disp=0)
print(f"\nLOGISTIC REGRESSION -- Selected model ({len(sig_feats)} features):")
print(logit_sel.summary2())

# Odds ratios + 95% CI for selected model
or_sel    = np.exp(logit_sel.params).drop('const')
or_ci_sel = np.exp(logit_sel.conf_int()).drop('const')
print("\nOdds Ratios (selected model):")
for feat in or_sel.index:
    print(f"  {feat}: OR={or_sel[feat]:.4f}  [{or_ci_sel.loc[feat,0]:.4f}, {or_ci_sel.loc[feat,1]:.4f}]")

# Odds ratio plot -- selected model
fig, ax = plt.subplots(figsize=(8, 5))
y_pos = range(len(or_sel))
ax.barh(list(y_pos), or_sel.values, color='steelblue', alpha=0.8)
for i, feat in enumerate(or_sel.index):
    ax.plot([or_ci_sel.loc[feat, 0], or_ci_sel.loc[feat, 1]], [i, i],
            color='black', lw=2, solid_capstyle='round')
ax.axvline(x=1, color='tomato', ls='--', lw=1.5, label='OR = 1 (no effect)')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(or_sel.index)
ax.set_xlabel('Odds Ratio')
ax.set_title('Odds Ratios -- Selected Original Model (95% CI)')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

### Logistic Regression with PCA Variables

Using the 3 retained principal components as predictors, all three are significant:

- PC1 (Physical Labour): OR = 1.608 [1.452, 1.780], p < 0.001
- PC2 (Desk Work): OR = 1.187 [1.070, 1.318], p = 0.001
- PC3 (Age/Seniority): OR = 1.173 [1.031, 1.334], p = 0.015

McFadden Pseudo-R² = 0.076, AIC = 1321.32, 4 parameters (3 PCs + intercept).

Interpreting the coefficients:

- PC1 (OR = 1.61): a one-standard-deviation increase in PC1 — moving toward a more physically demanding job profile — is associated with a 61% increase in the odds of chronic pain. This is the dominant risk factor, and the large OR reflects the combined effect of four correlated physical-work variables compressed into one dimension.
- PC2 (OR = 1.19): workers who score higher on the desk-work dimension (more computer time, more meetings, higher income) have 19% higher odds of chronic pain per SD increase. This likely reflects musculoskeletal strain from prolonged static postures and sedentary work.
- PC3 (OR = 1.17): older, more senior workers have 17% higher odds of chronic pain per SD increase. Age is an independent risk factor for musculoskeletal conditions regardless of job type.

In [ ]:
# ============================================================
# 3.8: Logistic Regression -- PCA Variables (3 retained PCs)
# ============================================================

X_pca_3 = sm.add_constant(pc_named)
logit_pca = sm.Logit(y_w, X_pca_3).fit(disp=0)
print(f"\n{'='*60}")
print("LOGISTIC REGRESSION -- PCA model (3 retained PCs):")
print(logit_pca.summary2())

or_pca    = np.exp(logit_pca.params).drop('const')
or_ci_pca = np.exp(logit_pca.conf_int()).drop('const')
print("\nOdds Ratios (PCA model):")
for feat in or_pca.index:
    print(f"  {feat}: OR={or_pca[feat]:.4f}  [{or_ci_pca.loc[feat,0]:.4f}, {or_ci_pca.loc[feat,1]:.4f}]")

# Odds ratio plot -- PCA model
fig, ax = plt.subplots(figsize=(7, 4))
y_pos = range(len(or_pca))
ax.barh(list(y_pos), or_pca.values, color='darkorange', alpha=0.8)
for i, feat in enumerate(or_pca.index):
    ax.plot([or_ci_pca.loc[feat, 0], or_ci_pca.loc[feat, 1]], [i, i],
            color='black', lw=2, solid_capstyle='round')
ax.axvline(x=1, color='tomato', ls='--', lw=1.5, label='OR = 1 (no effect)')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(or_pca.index)
ax.set_xlabel('Odds Ratio')
ax.set_title('Odds Ratios -- PCA Model (3 PCs, 95% CI)')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

### Model Comparison

Comparing the three logistic regression models:

- Full original model (9 features): 10 parameters, log-likelihood = -651.81, AIC = 1323.62, BIC = 1374.52, Pseudo R² = 0.083
- Selected original model (6 features): 7 parameters, log-likelihood = -654.35, AIC = 1322.70, BIC = 1358.33, Pseudo R² = 0.079
- PCA model (3 components): 4 parameters, log-likelihood = -656.66, AIC = 1321.32, BIC = 1341.68, Pseudo R² = 0.076

A few observations:

- The PCA model uses only 4 parameters compared to 7 (selected) or 10 (full) — a significant parsimony gain.
- AIC (lower is better): the PCA model achieves the best AIC (1321.32) despite the lowest pseudo-R². This is because AIC penalises for additional parameters, so the full model's slightly better log-likelihood is outweighed by having 6 extra parameters.
- Pseudo-R²: the full model has the highest McFadden R² (0.083), but all three are in a narrow range (0.076–0.083). None of the models explain a large fraction of deviance — chronic pain has a complex aetiology that 9 variables are unlikely to fully capture.
- Interpretability: the selected original model has a direct clinical interpretation — each coefficient describes the effect of a specific, measurable work characteristic. The PCA model is harder to translate into actionable advice: telling an employer "reduce PC1" is less useful than "reduce lifting hours." But the PCA model is more concise and less affected by multicollinearity.

Overall, the PCA model is preferable for predictive parsimony (best AIC, fewest parameters), while the selected original model is preferable for causal inference and workplace intervention (direct variable interpretation, though still subject to collinearity caveats).

In [ ]:
# ============================================================
# 3.9: Model Comparison
# ============================================================

comparison_df = pd.DataFrame({
    'Model':        ['Full original (9)', 'Selected original (6)', 'PCA (3 PCs)'],
    'N Parameters': [int(logit_full.df_model) + 1,
                     int(logit_sel.df_model)  + 1,
                     int(logit_pca.df_model)  + 1],
    'Log-Lik':      [round(logit_full.llf, 2), round(logit_sel.llf, 2), round(logit_pca.llf, 2)],
    'AIC':          [round(logit_full.aic, 2), round(logit_sel.aic, 2), round(logit_pca.aic, 2)],
    'BIC':          [round(logit_full.bic, 2), round(logit_sel.bic, 2), round(logit_pca.bic, 2)],
    'Pseudo R²':    [round(logit_full.prsquared, 4),
                     round(logit_sel.prsquared, 4),
                     round(logit_pca.prsquared, 4)],
})
print(f"\n{'='*60}")
print("MODEL COMPARISON:")
print(comparison_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Logistic Regression Model Comparison', fontsize=13)

models     = comparison_df['Model'].tolist()
aic_vals   = comparison_df['AIC'].tolist()
pr2_vals   = comparison_df['Pseudo R²'].tolist()

axes[0].bar(models, aic_vals, color=['steelblue', 'steelblue', 'darkorange'], alpha=0.85)
axes[0].set_ylabel('AIC')
axes[0].set_title('AIC (lower = better)')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim(min(aic_vals) - 5, max(aic_vals) + 5)
for i, v in enumerate(aic_vals):
    axes[0].text(i, v + 0.3, f'{v:.1f}', ha='center', fontsize=9)

axes[1].bar(models, pr2_vals, color=['steelblue', 'steelblue', 'darkorange'], alpha=0.85)
axes[1].set_ylabel("McFadden's Pseudo R²")
axes[1].set_title("Pseudo R² (higher = better)")
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim(0, max(pr2_vals) + 0.01)
for i, v in enumerate(pr2_vals):
    axes[1].text(i, v + 0.0005, f'{v:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## Q3. Reflection Questions

### Question 1: Why can PCs be predictive of chronic pain if PCA ignored the target variable?

PCA is an unsupervised method that finds directions of maximum variance in the feature space, completely ignoring the chronic_pain target. However, the principal components can still be predictive because the features themselves carry information about the outcome. PCA doesn't need to "know" the target to extract useful structure — if the original variables are predictive of chronic pain, and if that predictive information is correlated with the main directions of variation in the data, then the PCs will inherit that predictive power.

In this dataset, the first PC captures physical labour intensity (standing, lifting, manual work, repetitive motions), which happens to be the strongest driver of musculoskeletal chronic pain. PCA identified this dimension simply because physical-work variables vary together across workers — it didn't need the outcome to discover the cluster. The predictiveness of PC1 is a consequence of the fact that the signal driving chronic pain (physical workload) also happens to be the dominant axis of variation in the input data. If the signal were orthogonal to the main variance directions, PCA would fail to capture it and the PCs would be uninformative — but here the two coincide.

---

### Question 2: Is PCA guaranteed to improve predictive performance?

No — PCA is not guaranteed to improve predictive performance. A few reasons:

1. Discarded variance may contain signal: PCA retains directions of maximum variance, but the outcome may be correlated with low-variance components that get discarded. In such a case, PCA would throw away the most predictive information while keeping noise-heavy directions.
2. Unsupervised nature: because PCA ignores the target variable, there's no guarantee that the retained components are the most predictive ones — only that they explain the most variance. Supervised alternatives like Partial Least Squares explicitly maximise covariance between the PCs and the target and can outperform PCA for prediction.
3. Information loss: retaining 3 components out of 9 discards roughly 50% of total variance. Even if all 9 original variables were good predictors, the PCA model is working with a compressed and partially degraded representation.
4. In this analysis, the pseudo-R² of the PCA model (0.076) is slightly lower than the selected original model (0.079), confirming that some predictive information was lost in the compression. The AIC advantage of the PCA model comes from parsimony, not from fitting the data better.

PCA can help predictive performance in high-dimensional settings by reducing overfitting and multicollinearity, but it's not a universal improvement — its benefit depends on whether the variance structure aligns with the predictive signal.

---

### Question 3: Feature Selection vs Feature Extraction (PCA)

Feature selection and feature extraction are both dimensionality reduction strategies, but they operate differently:

- Feature selection keeps a subset of the original variables, unchanged. The output variables still have their original meaning. For example, keeping only lifting hours and age means we can directly interpret each coefficient.
- Feature extraction (PCA) creates entirely new synthetic variables — linear combinations of all original variables. The output dimensions (principal components) don't correspond to any single measured variable.

The key conceptual difference is that feature selection preserves the original variable space (choosing what to keep and what to discard), while feature extraction creates a new lower-dimensional representation (transforming all variables into derived components). In the original selected logistic regression model, we can directly say "each extra hour of lifting increases the log-odds by 0.278." With PCA, the coefficient for PC1 describes the effect of an abstract composite — interpretable in terms of the original variables only indirectly, via the loadings.